# 05 — Paired Native YOLO26n Training

Reuse proven equivalent five-class S10 Raw/v2 checkpoints or fine-tune both branches with native Ultralytics. This phase reads the saved P4 receipt and never prepares data or searches for a preprocessing policy.

- **Inputs:** P4 receipt, frozen protocol/subsets/v2 policy, pinned local COCO-80 YOLO26n and the historical #21 evidence when reusing old runs.
- **Outputs:** Native best.pt/last.pt, results.csv and args.yaml remain local; `docs/round2-training-v1/` receives checkpoint/provenance receipts, native histories and learning curves for #35.
- **Prerequisites:** `uv sync --locked --extra study` and the saved local inputs listed in section 1.

**Outline**

1. [Configuration and Prerequisites](#configuration-prerequisites)
2. [Paired Provenance and Reuse Decision](#paired-provenance)
3. [Raw and Frozen-v2 Native Runs](#native-runs)
4. [Convergence and Recorded Resource Costs](#convergence-costs)
5. [Checkpoint Summary and Development Handoff](#checkpoint-summary)

Set paths in section 1, then **Restart & Run All**. This notebook executes only its own phase. Missing or stale inputs fail; original data, policies, historical evidence and final-holdout pixels remain untouched.

<a id="configuration-prerequisites"></a>

## 1. Configuration and Prerequisites

Defaults validate and reuse the completed #21 runs. For a fresh paired study, use a new `TRAIN_ROOT` and `REPORT_ROOT`, set `HISTORICAL_PATH=None` and `ALLOW_TRAINING=True`. Native resume is explicit per branch; it is not bitwise continuation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vehicle_detection.common import freeze_json, read_json, require_paths
from vehicle_detection.finetuning import prepare_pair, train_branch
from vehicle_detection.protocol import validate_protocol
from vehicle_detection.training_preparation import validate_preparation
from vehicle_detection.training_report import training_evidence, verify_historical_reuse

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
PROTOCOL_PATH = ROOT / "runs/issue5-v1/protocol.json"
SUBSETS_PATH = ROOT / "runs/issue6-v2/subsets.json"
POLICY_PATH = ROOT / "docs/issue6-v2/policy.json"
DECISION_PATH = ROOT / "docs/issue11-v2/v3_decision.json"
PREPARED = ROOT / "runs/preprocessing-five-v1"
PREPARATION_PATH = ROOT / "docs/round2-training-v1/preparation.json"
CHECKPOINT = ROOT / "weights/yolo26n.pt"
TRAIN_ROOT = ROOT / "runs/training-five-s10-v1"
REPORT_ROOT = ROOT / "docs/round2-training-v1"
HISTORICAL_PATH = ROOT / "docs/issue21/s10_results.json"
ALLOW_TRAINING = False
RESUME = {"raw": False, "adaptive": False}
require_paths(
    protocol=PROTOCOL_PATH,
    subsets=SUBSETS_PATH,
    policy=POLICY_PATH,
    decision=DECISION_PATH,
    preparation=PREPARATION_PATH,
    checkpoint=CHECKPOINT,
)
if HISTORICAL_PATH is not None:
    require_paths(historical_evidence=HISTORICAL_PATH)
protocol = validate_protocol(read_json(PROTOCOL_PATH))
preparation = validate_preparation(read_json(PREPARATION_PATH), PREPARED)

<a id="paired-provenance"></a>

## 2. Paired Provenance and Reuse Decision

Authenticate historical pair/args/history/checkpoint hashes before reuse. The current pinned initialization, source IDs, exported image identities, taxonomy, v2 policy, seed, freeze and settings must match P4 and the historical run. A mismatch fails; it never silently launches replacement training.

In [ ]:
historical = read_json(HISTORICAL_PATH) if HISTORICAL_PATH is not None else None
if historical is not None:
    if any(RESUME.values()):
        raise ValueError("Historical runs are read-only; use a fresh training root")
    verify_historical_reuse(TRAIN_ROOT, historical)
if not ALLOW_TRAINING:
    if any(RESUME.values()):
        raise ValueError("Resume requires ALLOW_TRAINING=True")
    require_paths(
        **{
            f"{branch}_{name}": TRAIN_ROOT / branch / name
            for branch in ("raw", "adaptive")
            for name in ("args.yaml", "results.csv", "weights/best.pt", "weights/last.pt")
        }
    )
experiment = prepare_pair(
    protocol,
    read_json(SUBSETS_PATH),
    read_json(POLICY_PATH),
    read_json(DECISION_PATH),
    PREPARED / "datasets/S10",
    PREPARED / "development/raw",
    CHECKPOINT,
    TRAIN_ROOT,
)
if (
    any(experiment["training"][b] != preparation["exports"][b] for b in ("raw", "adaptive"))
    or experiment["development"] != preparation["exports"]["development"]
):
    raise ValueError("Training inputs differ from P4")
display(pd.Series(experiment["settings"], name="shared native settings"))
print("Frozen backbone modules:", experiment["freeze"])

<a id="native-runs"></a>

## 3. Raw and Frozen-v2 Native Runs

Each branch starts from the same pinned COCO-80 weights, with 20 epochs, batch 16, seed 2026 and freeze=11. Completed native runs are read without a trainer. An incomplete run fails unless explicitly resumed; both arms keep equal budgets.

In [ ]:
completed = {
    branch: train_branch(experiment, branch, TRAIN_ROOT, resume=RESUME[branch])
    for branch in ("raw", "adaptive")
}
handoff = training_evidence(experiment, preparation, completed, TRAIN_ROOT, historical=historical)
print(handoff["status"])
print("Paired budgets:", {b: r["budget"] for b, r in completed.items()})

<a id="convergence-costs"></a>

## 4. Convergence and Recorded Resource Costs

Plot native histories without inference. Mark Raw clock resets explicitly. CSV time includes training and epoch validation; setup, downtime, discarded partial epochs and final validation are excluded. Historical GPU memory peaks were not recorded. A single-seed result does not establish v2 superiority.

In [ ]:
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
plots = [
    ("metrics/mAP50-95(B)", "Raw development mAP50:95", 100),
    ("metrics/mAP50(B)", "Raw development mAP50", 100),
    ("train/box_loss", "Training box loss", 1),
    ("val/box_loss", "Raw development box loss", 1),
]
for branch in ("raw", "adaptive"):
    path = TRAIN_ROOT / branch / "results.csv"
    history = pd.read_csv(path)
    target = REPORT_ROOT / f"{branch}_results.csv"
    if target.exists() and target.read_bytes() != path.read_bytes():
        raise ValueError("Native history evidence changed; use a new report root")
    if not target.exists():
        target.write_bytes(path.read_bytes())
    for ax, (metric, title, factor) in zip(axes.flat, plots, strict=True):
        ax.plot(history.epoch, history[metric] * factor, label=branch)
        for epoch in completed[branch]["timing_reset_epochs"]:
            ax.axvline(epoch - 0.5, linestyle=":", alpha=0.6)
        ax.set(title=title, xlabel="Epoch", ylabel="AP (%)" if factor == 100 else "Loss")
        ax.legend()
        ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(REPORT_ROOT / "convergence.png", dpi=150)
plt.show()
display(
    pd.DataFrame(
        [
            {
                "branch": b,
                "best_epoch": r["best_epoch"],
                "final_epoch": r["final_epoch"],
                "training_validation_seconds": r["training_validation_seconds"],
                "timing_reset_epochs": r["timing_reset_epochs"],
            }
            for b, r in completed.items()
        ]
    )
)

<a id="checkpoint-summary"></a>

## 5. Checkpoint Summary and Development Handoff

All four native checkpoints must load with five-class heads and match saved settings, native history metrics and historical hashes on reuse. #35 receives their exact paths/hashes. Raw resumed at epoch 13; the selected best checkpoints precede that interruption. No v3, S20 or final-test evaluation is authorized by this handoff.

In [ ]:
freeze_json(REPORT_ROOT / "training.json", handoff)
print("P5 identity:", handoff["handoff_id"])
for branch, receipt in handoff["branches"].items():
    print(branch, receipt["checkpoints"], "interruption:", receipt["interruption"])
print("Next: #35 development evaluation of Raw and frozen v2.")